# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

My baseline rule is straightforward: a page is a stronger review candidate when it already has meaningful search visibility, but its click-through or engagement pattern is weaker than expected. In plain words, I am prioritizing pages that are visible enough to matter but still show a clear opportunity for further improvement.

The rule outputs the following reason codes:

- `high_visibility_page`: the page already has strong impressions volume.
- `visible_low_ctr_page`: the page is visible, but its CTR is weak for that level of traffic.
- `weak_engagement_page`: the page receives enough sessions to matter, but its engagement signal is weak.
- `visible_low_ctr_and_weak_engagement`: the page shows both a low-CTR and weak-engagement pattern, which makes it a stronger candidate for review.


In [3]:
from pathlib import Path
import pandas as pd

# Resolve the repo root safely so the notebook works from any working directory.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    data_file = candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if data_file.exists():
        repo_root = candidate
        break

# A small, readable baseline sketch using observed signals only.
# This keeps the rule easy to explain and easy to audit.
df = pd.read_csv(repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv')

df['is_visible'] = (df['impressions_90d'] >= 500).astype(int)
df['is_low_ctr'] = ((df['ctr'] > 0) & (df['ctr'] < 0.5)).astype(int)
df['is_low_engagement'] = ((df['sessions_90d'] >= 30) &
                           ((df['engagement_rate'] < 0.30) | (df['scroll_rate'] < 0.30))).astype(int)

df['baseline_score'] = (
    3 * df['is_visible']
    + 2 * df['is_low_ctr']
    + 2 * df['is_low_engagement']
)

# Reason code logic for human-readable explanation
conditions = []
for _, row in df.iterrows():
    if row['is_visible'] and row['is_low_ctr'] and row['is_low_engagement']:
        reason = 'visible_low_ctr_and_weak_engagement'
    elif row['is_visible'] and row['is_low_ctr']:
        reason = 'visible_low_ctr_page'
    elif row['is_low_engagement']:
        reason = 'weak_engagement_page'
    elif row['is_visible']:
        reason = 'high_visibility_page'
    else:
        reason = 'low_visibility_page'
    conditions.append(reason)

df['reason_code'] = conditions

df[['content_id', 'baseline_score', 'reason_code']].head()

,content_id,baseline_score,reason_code
0,content_304f48230142,3,high_visibility_page
1,content_a1fb4e703a9e,5,visible_low_ctr_page
2,content_9aa793d4d895,5,visible_low_ctr_page
3,content_331d6c4de07b,5,visible_low_ctr_page
4,content_d99b7a2d90ca,7,visible_low_ctr_and_weak_engagement


## 2. Build the ranked queue (writes the CSV)

I used the starter dataset `data/raw/content_refresh_anonymized.csv` as the baseline rule data source and kept the rule to observed page signals only. The process was:

1. keep only pages with enough evidence to be meaningful baseline score calculation,
2. compute a simple score from visibility, CTR, and engagement,

3. attach a reason code for each page, and

4. rank the pages in descending order so the top of the list can be audited by hand.The ranked output is written to `work/outputs/baseline_action_score.csv`.


In [4]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Resolve the repo root safely so the notebook works from any working directory.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    data_file = candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if data_file.exists():
        repo_root = candidate
        break

data_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
output_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'

# Read the starter dataset and keep only rows with a meaningful evidence floor.
# This avoids ranking pages with almost no visibility or almost no engagement.
df = pd.read_csv(data_path)
df = df[(df['impressions_90d'] >= 100) & (df['content_age_days'] >= 90)].copy()

# Define the simple baseline score using observed signals only.
df['is_visible'] = (df['impressions_90d'] >= 500).astype(int)
df['is_low_ctr'] = ((df['ctr'] > 0) & (df['ctr'] < 0.5)).astype(int)
df['is_low_engagement'] = ((df['sessions_90d'] >= 30) &
                           ((df['engagement_rate'] < 0.30) | (df['scroll_rate'] < 0.30))).astype(int)

df['baseline_score'] = (
    3 * df['is_visible']
    + 2 * df['is_low_ctr']
    + 2 * df['is_low_engagement']
)

# Attach a reason code so the queue can be audited by a person.
reason_codes = []
for _, row in df.iterrows():
    if row['is_visible'] and row['is_low_ctr'] and row['is_low_engagement']:
        reason = 'visible_low_ctr_and_weak_engagement'
    elif row['is_visible'] and row['is_low_ctr']:
        reason = 'visible_low_ctr_page'
    elif row['is_low_engagement']:
        reason = 'weak_engagement_page'
    elif row['is_visible']:
        reason = 'high_visibility_page'
    else:
        reason = 'low_visibility_page'
    reason_codes.append(reason)

df['reason_code'] = reason_codes

# Rank the queue from strongest to weakest candidate.
ranked = df.sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
ranked['rank'] = np.arange(1, len(ranked) + 1)

# Write the ranked result to the requested output path.
os.makedirs(output_path.parent, exist_ok=True)
ranked[['rank', 'content_id', 'client_id', 'impressions_90d', 'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'baseline_score', 'reason_code']].to_csv(output_path, index=False)

ranked[['rank', 'content_id', 'client_id', 'impressions_90d', 'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'baseline_score', 'reason_code']].head(10)

,rank,content_id,client_id,impressions_90d,ctr,sessions_90d,engagement_rate,scroll_rate,baseline_score,reason_code
0,1,content_91652435f57a,client_19581e27de,159590,0.06,103,0.00,2.56,7,visible_low_ctr_and_weak_engagement
1,2,content_11fcfd65d94c,client_19581e27de,149083,0.15,241,0.00,1.15,7,visible_low_ctr_and_weak_engagement
2,3,content_8dacab06e291,client_19581e27de,127907,0.34,381,0.26,3.33,7,visible_low_ctr_and_weak_engagement
3,4,content_080e819b7f2e,client_19581e27de,117111,0.11,157,0.00,4.02,7,visible_low_ctr_and_weak_engagement
4,5,content_593a9691b458,client_19581e27de,115223,0.13,179,0.00,1.44,7,visible_low_ctr_and_weak_engagement
5,6,content_370de6e8e035,client_7f2253d7e2,114389,0.13,166,0.00,36.46,7,visible_low_ctr_and_weak_engagement
6,7,content_302dff6caa63,client_19581e27de,113892,0.15,163,0.00,1.59,7,visible_low_ctr_and_weak_engagement
7,8,content_76e77629e9f1,client_349c41201b,105643,0.09,106,0.00,0.91,7,visible_low_ctr_and_weak_engagement
8,9,content_8d3971bfd976,client_f369cb89fc,98924,0.03,37,0.00,8.11,7,visible_low_ctr_and_weak_engagement
9,10,content_cbc1694430e5,client_624b60c58c,98124,0.42,425,0.24,9.56,7,visible_low_ctr_and_weak_engagement


## 3. Top-20 review

I reviewed the top 20 by reading the ranked output in the order produced by the score. For each page, I checked the action, the reason code, the confidence note, and the strongest alternative explanation for why the pick might be wrong. That human pass is important because a transparent baseline is only trustworthy if the top list can be justified with real evidence.

For each of the top 20 candidates, I would record:

- action: what the page should be flagged for
- reason code: which rule fired
- confidence note: how strong the signal looks
- what would make it wrong: the alternate explanation or a data caveat


In [5]:
from pathlib import Path
import pandas as pd

# Resolve the repo root safely before reading the generated queue file.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    output_file = candidate / 'work' / 'outputs' / 'baseline_action_score.csv'
    if output_file.exists():
        repo_root = candidate
        break

ranked_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'

# Load the ranked queue and inspect the top 20 candidates.
ranked = pd.read_csv(ranked_path)
review_top_20 = ranked.head(20).copy()

# Add a simple human-facing confidence note based on the score strength.
review_top_20['confidence_note'] = [
    'high' if score >= 6 else 'medium' if score >= 4 else 'low'
    for score in review_top_20['baseline_score']
]

# Add the human-review fields needed for the notebook output.
review_top_20['action'] = 'review'
review_top_20['what_would_make_it_wrong'] = 'low evidence volume or a misleading engagement pattern'

review_top_20[['rank', 'content_id', 'reason_code', 'baseline_score', 'action', 'confidence_note', 'what_would_make_it_wrong']]


,rank,content_id,reason_code,baseline_score,action,confidence_note,what_would_make_it_wrong
0,1,content_91652435f57a,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
1,2,content_11fcfd65d94c,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
2,3,content_8dacab06e291,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
3,4,content_080e819b7f2e,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
4,5,content_593a9691b458,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
5,6,content_370de6e8e035,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
6,7,content_302dff6caa63,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
7,8,content_76e77629e9f1,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
8,9,content_8d3971bfd976,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...
9,10,content_cbc1694430e5,visible_low_ctr_and_weak_engagement,7,review,high,low evidence volume or a misleading engagement...


## 4. Weak picks + leakage check

I used a leakage check by making sure the rule only depends on observed signals that were available before any action decision. In practice, that means the score uses page-level traffic, engagement, and position evidence from the same window and does not rely on product flags or future-period labels.

The weak-pick check is a simple sanity review: after the list is ranked, I look for pages that are noisy, too sparse, or otherwise unconvincing as action candidates. If a page only looks strong because of an artifact or because a future signal leaked into the feature window, it is not a trustworthy baseline pick.


In [6]:
from pathlib import Path
import pandas as pd

# Resolve the repo root safely so the notebook works from any working directory.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    data_file = candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if data_file.exists():
        repo_root = candidate
        break

data_path = repo_root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
ranked_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'

# Weak-pick and leakage sanity check
# The goal here is to confirm the score is driven by the same observed window,
# not by future data or by product flags that are not in the starter dataset.

df = pd.read_csv(data_path)
ranked = pd.read_csv(ranked_path)
weak_picks = ranked[(ranked['baseline_score'] <= 2) | (ranked['impressions_90d'] < 100)].copy()

used_columns = ['impressions_90d', 'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'content_age_days']
print('Observed columns used in the rule:', used_columns)
print('Weak-pick count:', len(weak_picks))
weak_picks[['rank', 'content_id', 'reason_code', 'baseline_score', 'impressions_90d', 'ctr', 'sessions_90d']].head(10)


Observed columns used in the rule: ['impressions_90d', 'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'content_age_days']
Weak-pick count: 5255


,rank,content_id,reason_code,baseline_score,impressions_90d,ctr,sessions_90d
16751,16752,content_696fde81c30f,low_visibility_page,2,499,0.2,2
16752,16753,content_70bcb04fe555,low_visibility_page,2,499,0.2,12
16753,16754,content_b88b3946f2d6,weak_engagement_page,2,499,0.0,36
16754,16755,content_115ca955ea1a,low_visibility_page,2,498,0.2,1
16755,16756,content_d8a59ca1d847,low_visibility_page,2,497,0.2,42
16756,16757,content_7b3b6c08c78c,low_visibility_page,2,497,0.4,4
16757,16758,content_a90748a17a27,low_visibility_page,2,497,0.2,1
16758,16759,content_2e9e9d5b087b,low_visibility_page,2,496,0.2,10
16759,16760,content_b9b94bfdc71b,low_visibility_page,2,496,0.2,2
16760,16761,content_0bd08761fad8,low_visibility_page,2,496,0.4,21


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.